# PICRUSt2: final nine-ASV KO analysis (Supplementary Tables S5–S7)

This notebook reproduces the **downstream KO occurrence summaries** in the revised IJMS manuscript from existing Nephele/PICRUSt2 predictions. It selects only the nine ASVs assigned to the three LR-prioritized taxa. The other 39 ASVs in the source KO file are not part of these final supplementary tables.

Inputs: `informative_taxa_KO_predicted.csv` and `ASV_dna_sequences_taxonomy.xlsx`. When available, `pre_aligned_sequences.fasta` verifies exact DNA-to-ASV matches and the original `PipelineResults_5677a27dd332.tar` archive verifies sample-weighted NSTI.

The notebook *does not rerun PICRUSt2* from the FASTA/BIOM inputs. Its expected counts are used as independent assertions, not to generate the calculated results.

In [ ]:
# Mount Google Drive only when running in Colab.
import sys
from pathlib import Path
if 'google.colab' in sys.modules and not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

import gzip
import hashlib
import io
import json
import os
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd

# Input directories (Google Colab Drive or an uploaded local copy).
DEFAULT_BASE = Path('/content/drive/MyDrive/project/picrust2')
BASE = Path(os.environ.get('PICRUST2_BASE', str(DEFAULT_BASE)))
if not BASE.exists():
    BASE = Path('/mnt/data')
OUTPUT = Path(os.environ.get('PICRUST2_OUTPUT', str(BASE / 'PICRUSt2_9ASV_manuscript_outputs')))

EXPECTED_TAXA = {
    'Lachnospiraceae_ND3007_group': 4,
    'Dielma': 3,
    'Rhizobiales_unclassified': 2,
}

# Table entries and annotations follow the revised manuscript's S5-S7 tables.
# The integer on each row is a check against the published occurrence count;
# it is never used to calculate an output count.
TABLES = {
    'S5': [
        ('Lachnospiraceae_ND3007_group', 'K01442', 'Choloylglycine hydrolase [EC:3.5.1.24]', 4),
        ('Lachnospiraceae_ND3007_group', 'K07007', '3-dehydro-bile acid delta(4,6)-reductase [EC:1.3.1.114]', 4),
        ('Dielma', 'K03453', 'Bile acid:Na+ symporter', 3),
        ('Rhizobiales_unclassified', 'K01442', 'Choloylglycine hydrolase [EC:3.5.1.24]', 1),
        ('Rhizobiales_unclassified', 'K07007', '3-dehydro-bile acid delta(4,6)-reductase [EC:1.3.1.114]', 2),
        ('Rhizobiales_unclassified', 'K03453', 'Bile acid:Na+ symporter', 1),
    ],
    'S6': [
        ('Lachnospiraceae_ND3007_group', 'K01181', 'Endo-1,4-beta-xylanase [EC:3.2.1.8]', 4),
        ('Lachnospiraceae_ND3007_group', 'K01208', 'Cyclomaltodextrinase / maltogenic alpha-amylase / neopullulanase', 4),
        ('Lachnospiraceae_ND3007_group', 'K01190', 'Beta-galactosidase [EC:3.2.1.23]', 4),
        ('Lachnospiraceae_ND3007_group', 'K07407', 'Alpha-galactosidase [EC:3.2.1.22]', 4),
        ('Dielma', 'K02755', 'Beta-glucoside PTS system EIIA component', 3),
        ('Dielma', 'K02757', 'Beta-glucoside PTS system EIICBA component', 3),
        ('Dielma', 'K02759', 'Cellobiose PTS system EIIA component', 3),
        ('Dielma', 'K02760', 'Cellobiose PTS system EIIB component', 3),
        ('Dielma', 'K02761', 'Cellobiose PTS system EIIC component', 3),
        ('Dielma', 'K01223', '6-phospho-beta-glucosidase [EC:3.2.1.86]', 3),
        ('Rhizobiales_unclassified', 'K01198', 'Xylan 1,4-beta-xylosidase [EC:3.2.1.37]', 1),
        ('Rhizobiales_unclassified', 'K05349', 'Beta-glucosidase [EC:3.2.1.21]', 2),
    ],
    'S7': [
        ('Dielma', 'K00826', 'Branched-chain amino acid aminotransferase', 3),
        ('Dielma', 'K00625', 'Phosphate acetyltransferase [EC:2.3.1.8]', 3),
        ('Dielma', 'K15024', 'Putative phosphotransacetylase [EC:2.3.1.8]', 3),
        ('Dielma', 'K00925', 'Acetate kinase [EC:2.7.2.1]', 3),
        ('Dielma', 'K00626', 'Acetyl-CoA C-acetyltransferase [EC:2.3.1.9]', 3),
        ('Dielma', 'K00634', 'Phosphate butyryltransferase [EC:2.3.1.19]', 3),
        ('Dielma', 'K00929', 'Butyrate kinase [EC:2.7.2.7]', 3),
        ('Lachnospiraceae_ND3007_group', 'K00826', 'Branched-chain amino acid aminotransferase', 4),
        ('Lachnospiraceae_ND3007_group', 'K00625', 'Phosphate acetyltransferase [EC:2.3.1.8]', 4),
        ('Lachnospiraceae_ND3007_group', 'K00925', 'Acetate kinase [EC:2.7.2.1]', 4),
        ('Rhizobiales_unclassified', 'K00826', 'Branched-chain amino acid aminotransferase', 2),
        ('Rhizobiales_unclassified', 'K00166', '2-oxoisovalerate dehydrogenase E1 alpha subunit', 1),
        ('Rhizobiales_unclassified', 'K09699', '2-oxoisovalerate dehydrogenase E2 component', 1),
        ('Rhizobiales_unclassified', 'K00625', 'Phosphate acetyltransferase [EC:2.3.1.8]', 1),
        ('Rhizobiales_unclassified', 'K00925', 'Acetate kinase [EC:2.7.2.1]', 2),
        ('Rhizobiales_unclassified', 'K00626', 'Acetyl-CoA C-acetyltransferase [EC:2.3.1.9]', 2),
        ('Rhizobiales_unclassified', 'K00074', '3-hydroxybutyryl-CoA dehydrogenase [EC:1.1.1.157]', 2),
        ('Rhizobiales_unclassified', 'K00248', 'Butyryl-CoA dehydrogenase [EC:1.3.8.1]', 2),
        ('Rhizobiales_unclassified', 'K11264', 'Methylmalonyl-CoA decarboxylase [EC:4.1.1.-]', 1),
    ],
}
DIRECT_ISOBUTYRATE_ROUTE = ('K00186', 'K00187', 'K00188', 'K00189')

## Load and validate the original input files

Source taxon counts are checked against the full 6,144-ASV taxonomy table. Exact ASV ID matching is additionally checked if the original FASTA is provided.

In [ ]:
def locate_input(env_name: str, candidates: list[str], required: bool = True) -> Path | None:
    if os.environ.get(env_name):
        p = Path(os.environ[env_name])
        if not p.is_file():
            raise FileNotFoundError(f'{env_name}={p}')
        return p
    dirs = [BASE, BASE / 'inputs', BASE / 'analysis', BASE / 'outputs', Path('/mnt/data'), Path.cwd()]
    for directory in dirs:
        for name in candidates:
            p = directory / name
            if p.is_file():
                return p
    if required:
        raise FileNotFoundError(f'{env_name}: cannot locate {candidates}; set the environment variable.')
    return None


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def taxonomy_masks(df: pd.DataFrame) -> dict[str, pd.Series]:
    genus = df['Genus'].astype('string').str.strip()
    order = df['Order'].astype('string').str.strip()
    missing_genus = genus.isna() | genus.str.lower().isin(['', 'unassigned', 'unclassified', 'nan']).fillna(False)
    return {
        'Lachnospiraceae_ND3007_group': genus.eq('Lachnospiraceae_ND3007_group').fillna(False),
        'Dielma': genus.eq('Dielma').fillna(False),
        'Rhizobiales_unclassified': order.eq('Rhizobiales').fillna(False) & missing_genus,
    }


def read_fasta_sequences(path: Path) -> dict[str, str]:
    result = {}
    name, chunks = None, []
    with path.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if name is not None:
                    result[''.join(chunks).upper()] = name
                name, chunks = line[1:].split()[0], []
            elif line:
                chunks.append(line)
    if name is not None:
        result[''.join(chunks).upper()] = name
    return result


def check_taxonomy(path: Path, selected: pd.DataFrame, fasta_path: Path | None) -> dict:
    df = pd.read_excel(path, sheet_name='ASV_sequences')
    needed = {'DNA sequence', 'Genus', 'Order'}
    if not needed.issubset(df.columns):
        raise ValueError(f'ASV_sequences missing {sorted(needed - set(df.columns))}')
    if len(df) != 6144:
        raise AssertionError(f'Unexpected taxonomy ASV count: {len(df)} (expected 6144)')
    counts = {t: int(mask.sum()) for t, mask in taxonomy_masks(df).items()}
    if counts != EXPECTED_TAXA:
        raise AssertionError(f'Taxonomy ASV counts differ: {counts} vs {EXPECTED_TAXA}')
    result = {'taxonomy_rows': len(df), 'taxonomy_counts': counts,
              'dna_to_asv_mapping_verified': False}
    if fasta_path:
        seq_to_id = read_fasta_sequences(fasta_path)
        if len(seq_to_id) != 6144:
            raise AssertionError(f'Unexpected FASTA unique-sequence count: {len(seq_to_id)}')
        mappings = []
        for taxon, mask in taxonomy_masks(df).items():
            for seq in df.loc[mask, 'DNA sequence'].astype(str):
                asv_id = seq_to_id.get(seq.strip().upper())
                if asv_id is None:
                    raise AssertionError(f'No FASTA sequence match for {taxon}')
                mappings.append((taxon, asv_id))
        from_taxonomy = set(mappings)
        from_ko = set(selected[['informative_taxon', 'sequence']].itertuples(index=False, name=None))
        if from_taxonomy != from_ko:
            raise AssertionError(f'FASTA and KO taxon-ASV mappings differ: {from_taxonomy ^ from_ko}')
        result['dna_to_asv_mapping_verified'] = True
    return result


def read_optional_nsti(archive_path: Path | None) -> dict | None:
    if archive_path is None:
        return None
    with tarfile.open(archive_path, mode='r:*') as tar:
        members = [m for m in tar.getmembers() if m.name.endswith('/KO_metagenome_out/weighted_nsti.tsv.gz')]
        if len(members) != 1:
            raise ValueError(f'Expected one weighted_nsti.tsv.gz member, found {len(members)}')
        f = tar.extractfile(members[0])
        if f is None:
            raise ValueError('Cannot extract weighted NSTI file')
        nsti = pd.read_csv(io.BytesIO(gzip.decompress(f.read())), sep='\t')
    if 'sample' not in nsti.columns or 'weighted_NSTI' not in nsti.columns:
        raise ValueError('Unexpected weighted NSTI column names')
    vals = pd.to_numeric(nsti['weighted_NSTI'], errors='raise')
    if len(vals) != 88 or nsti['sample'].nunique() != 88:
        raise AssertionError('Expected weighted NSTI for 88 distinct samples')
    summary = {'n': int(len(vals)), 'mean': float(vals.mean()), 'median': float(vals.median()),
               'minimum': float(vals.min()), 'maximum': float(vals.max())}
    if not (round(summary['mean'], 4) == .0416 and round(summary['median'], 4) == .0355
            and round(summary['minimum'], 4) == .0153 and round(summary['maximum'], 4) == .1245):
        raise AssertionError(f'NSTI differs from the manuscript: {summary}')
    return summary

## Extract manuscript KO rows and verify counts

Positive predicted KO abundance (`> 0`) counts as predicted presence. The notebook outputs S5, S6 and S7 CSV tables and stops with an error if any published count differs.

In [ ]:
def run() -> dict:
    ko_path = locate_input('PICRUST2_KO_CSV', [
        'informative_taxa_KO_predicted.csv', 'informative_taxa_KO_predicted(1).csv',
        'informative_taxa_KO_predicted(2).csv'])
    taxonomy_path = locate_input('PICRUST2_TAXONOMY_XLSX', [
        'ASV_dna_sequences_taxonomy.xlsx', 'ASV_dna_sequences_taxonomy (2)(1).xlsx',
        'ASV_dna_sequences_taxonomy (2)(2).xlsx'])
    fasta_path = locate_input('PICRUST2_FASTA', ['pre_aligned_sequences.fasta'], required=False)
    archive_path = locate_input('PICRUST2_NEPHELE_ARCHIVE', [
        'PipelineResults_5677a27dd332.tar', 'PipelineResults_5677a27dd332.tar.gz'], required=False)

    ko = pd.read_csv(ko_path, low_memory=False)
    needed = {'sequence', 'informative_taxon'}
    if not needed.issubset(ko.columns):
        raise ValueError(f'KO CSV missing: {sorted(needed - set(ko.columns))}')
    if ko['sequence'].duplicated().any():
        raise ValueError('Duplicate ASV IDs in KO source data')
    selected = ko.loc[ko['informative_taxon'].isin(EXPECTED_TAXA)].copy()
    observed_counts = selected.groupby('informative_taxon').size().to_dict()
    if observed_counts != EXPECTED_TAXA or len(selected) != 9:
        raise AssertionError(f'Expected exactly nine selected ASVs; found {observed_counts}')
    print('Nine selected ASVs:', observed_counts)

    taxonomy_check = check_taxonomy(taxonomy_path, selected, fasta_path)
    all_kos = sorted({ko for rows in TABLES.values() for _, ko, _, _ in rows} | set(DIRECT_ISOBUTYRATE_ROUTE))
    missing = [k for k in all_kos if 'ko:' + k not in selected.columns]
    if missing:
        raise ValueError(f'KO source lacks columns: {missing}')
    cols = ['ko:' + k for k in all_kos]
    selected[cols] = selected[cols].apply(pd.to_numeric, errors='raise')
    if not np.isfinite(selected[cols].to_numpy(dtype=float)).all() or (selected[cols] < 0).any().any():
        raise ValueError('Missing, nonfinite or negative selected KO predictions')

    OUTPUT.mkdir(parents=True, exist_ok=True)
    mapping = selected[['informative_taxon', 'sequence']].rename(columns={
        'informative_taxon': 'Taxon', 'sequence': 'ASV_ID'}).sort_values(['Taxon', 'ASV_ID'])
    mapping.to_csv(OUTPUT / 'mapped_9_ASVs.csv', index=False)

    table_sizes = {}
    for name, entries in TABLES.items():
        out = []
        for taxon, ko_id, annotation, expected_positive in entries:
            block = selected.loc[selected['informative_taxon'].eq(taxon)]
            ids = block.loc[block['ko:' + ko_id].gt(0), 'sequence'].astype(str).tolist()
            observed = len(ids)
            if observed != expected_positive:
                raise AssertionError(f'{name} {taxon} {ko_id}: calculated {observed}/{len(block)}, '
                                     f'manuscript {expected_positive}/{EXPECTED_TAXA[taxon]}')
            out.append({'Taxon': taxon, 'KO': ko_id, 'KEGG_annotation': annotation,
                        'ASVs_with_KO': observed, 'total_ASVs': len(block),
                        'ASVs_with_KO_over_total': f'{observed}/{len(block)}',
                        'ASV_IDs_with_predicted_KO': ';'.join(ids)})
        pd.DataFrame(out).to_csv(OUTPUT / f'Supplementary_Table_{name}_verified.csv', index=False)
        table_sizes[name] = len(out)
        print(f'Supplementary Table {name}: PASS ({len(out)} KO occurrence rows)')

    isobuty = {}
    for ko_id in DIRECT_ISOBUTYRATE_ROUTE:
        positive_count = int(selected['ko:' + ko_id].gt(0).sum())
        if positive_count != 0:
            raise AssertionError(f'The direct-route KO {ko_id} is present in {positive_count} ASVs')
        isobuty[ko_id] = positive_count
    print('Direct isobutyryl-CoA route K00186-K00189: absent among nine ASVs (PASS)')

    # An ASV-level predicted KO is not evidence of expressed enzyme activity.
    summary = {
        'status': 'PASS: all curated manuscript KO occurrence counts match',
        'input_KO_CSV': str(ko_path), 'input_KO_CSV_sha256': sha256(ko_path),
        'input_taxonomy_XLSX': str(taxonomy_path), 'input_taxonomy_XLSX_sha256': sha256(taxonomy_path),
        'original_KO_source_ASVs': int(len(ko)), 'target_ASVs': 9,
        'target_ASV_counts': observed_counts, 'curated_row_counts': table_sizes,
        'taxonomic_crosscheck': taxonomy_check, 'direct_isobutyrate_route_KO_positive_ASVs': isobuty,
        'fasta_checked': str(fasta_path) if fasta_path else None,
        'weighted_NSTI': read_optional_nsti(archive_path),
        'Nephele_archive_checked': str(archive_path) if archive_path else None,
        'important_scope': 'Downstream KO extraction from stored PICRUSt2 predictions, not a new PICRUSt2 run.',
    }
    (OUTPUT / 'manuscript_validation.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print('Results saved to:', OUTPUT)
    if not fasta_path:
        print('Note: FASTA absent; DNA-to-ASV ID mapping was not independently rechecked.')
    if not archive_path:
        print('Note: Nephele archive absent; sample-weighted NSTI was not recalculated here.')
    return summary


summary = run()
print(json.dumps(summary, indent=2))